In [1]:
import pandas as pd
import numpy as np

In [2]:
books = pd.read_csv('books_data.csv')
reviews = pd.read_csv('Books_rating.csv')

In [3]:
#For books Need Title, authors, categories for analysis

books_critical = books.dropna(subset=['Title', 'authors', 'categories']).copy()
print(f"BOOKS - Keeping rows with Title, authors, categories:")
print(f"  Original: {len(books):,} rows")
print(f"  After drop: {len(books_critical):,} rows")
print(f"  Kept: {len(books_critical)/len(books)*100:.1f}%")

BOOKS - Keeping rows with Title, authors, categories:
  Original: 212,404 rows
  After drop: 165,744 rows
  Kept: 78.0%


In [4]:
#Drop other columns 

books_critical=books_critical.drop(columns= ['publisher', 'image', 'previewLink','infoLink', 'description','ratingsCount'])

In [5]:
#For reviews Need Title, Price, review_score

reviews_critical = reviews.dropna(subset=['Title', 'Price', 'review/score']).copy()
print(f"REVIEWS - Keeping rows with Title, Price, review_score:")
print(f"  Original: {len(reviews):,} rows")
print(f"  After drop: {len(reviews_critical):,} rows")
print(f"  Kept: {len(reviews_critical)/len(reviews)*100:.1f}%")

REVIEWS - Keeping rows with Title, Price, review_score:
  Original: 3,000,000 rows
  After drop: 481,164 rows
  Kept: 16.0%


In [6]:
reviews_critical=reviews_critical.drop(columns=['Id','User_id', 'profileName', 'review/time', 'review/helpfulness', 'review/summary', 'review/text'])

In [7]:
print("=== BOOKS DATA SHAPE ===")
print(f"Rows: {books_critical.shape[0]}, Columns: {books_critical.shape[1]}")
print("\n=== REVIEWS DATA SHAPE ===")
print(f"Rows: {reviews_critical.shape[0]}, Columns: {reviews_critical.shape[1]}")

# Check null percentages for EVERY column
print(" NULL PERCENTAGES IN BOOKS DATA:")
null_percentage_books = (books_critical.isnull().sum() / len(books_critical)) * 100
print(null_percentage_books[null_percentage_books > 0].sort_values(ascending=False))

print(" NULL PERCENTAGES IN REVIEWS DATA:")
null_percentage_reviews = (reviews_critical.isnull().sum() / len(reviews_critical)) * 100
print(null_percentage_reviews[null_percentage_reviews > 0].sort_values(ascending=False))

=== BOOKS DATA SHAPE ===
Rows: 165744, Columns: 4

=== REVIEWS DATA SHAPE ===
Rows: 481164, Columns: 3
 NULL PERCENTAGES IN BOOKS DATA:
publishedDate    0.263056
dtype: float64
 NULL PERCENTAGES IN REVIEWS DATA:
Series([], dtype: float64)


In [8]:
def clean_price(reviews_df):
    
    books_price = reviews_df.copy()
    # Convert price from string to float
    books_price['Price'] = books_price['Price'].replace('[\$,]', '', regex=True)
    books_price['Price'] = pd.to_numeric(books_price['Price'], errors='coerce')
    
    # Keep ONLY books with valid prices for pricing analysis
    books_price_clean = books_price[books_price['Price'].notna()].copy()
    print(f"Price Analysis Table: {len(books_price_clean):,} books with prices")
    
    return books_price_clean
        
# Clean Price Column
reviews_clean = clean_price(reviews_critical)

Price Analysis Table: 481,164 books with prices


In [9]:
def clean_review_scores(df):
    
    df_clean = df.copy()
    
    initial_count = len(df_clean)
    print(f"📊 Initial: {initial_count:,} reviews")
    
    # Remove invalid scores (only keep 0-5)
    invalid_mask = (df_clean['review/score'] < 0) | (df_clean['review/score'] > 5)
    invalid_reviews = df_clean[invalid_mask]
    df_clean = df_clean[~invalid_mask]
    
    print(f"✅ Removed {len(invalid_reviews):,} reviews with invalid scores")
    
    if len(invalid_reviews) > 0:
        print(f"   Invalid scores found: {invalid_reviews['review/score'].unique()}")
    
    print(f"📈 Final: {len(df_clean):,} reviews ({(len(df_clean)/initial_count*100):.1f}% kept)")
    print(f"   Score range: {df_clean['review/score'].min():.1f}-{df_clean['review/score'].max():.1f} stars")
    
    return df_clean


In [10]:
# Clean review scores Column
reviews_clean = clean_review_scores(reviews_clean)

📊 Initial: 481,164 reviews
✅ Removed 0 reviews with invalid scores
📈 Final: 481,164 reviews (100.0% kept)
   Score range: 1.0-5.0 stars


In [11]:
reviews_clean

,Title,Price,review/score
10,Wonderful Worship in Smaller Churches,19.40,5.0
11,Wonderful Worship in Smaller Churches,19.40,5.0
12,Wonderful Worship in Smaller Churches,19.40,5.0
13,Wonderful Worship in Smaller Churches,19.40,5.0
14,Whispers of the Wicked Saints,10.95,1.0
...,...,...,...
2999953,Very Bad Deaths: Library Edition,90.00,4.0
2999954,Very Bad Deaths: Library Edition,90.00,5.0
2999955,Very Bad Deaths: Library Edition,90.00,3.0
2999956,Very Bad Deaths: Library Edition,90.00,5.0


In [12]:
def clean_author_names(author_string):
    """Simpler version for standard 'First Last' format in lists"""
    if pd.isna(author_string) or author_string == "[]" or author_string == "['']":
        return None
    
    # Remove the brackets and quotes
    clean_name = str(author_string).strip("[]'\" ")
    
    # If multiple authors, take first one
    if ', ' in clean_name and "'" in clean_name:
        # Looks like: "'Author1', 'Author2'"
        authors = clean_name.split("', '")
        clean_name = authors[0].strip("'")
    
    # Remove titles
    titles = ['dr.', 'dr ', 'prof.', 'prof ', 'mr.', 'mr ', 'mrs.', 'mrs ', 'ms.', 'ms ']
    for title in titles:
        if clean_name.lower().startswith(title):
            clean_name = clean_name[len(title):].strip()
    
    # Get first name (for gender inference)
    first_name = clean_name.split()[0] if clean_name.split() else None
    
    return first_name

In [13]:
import re

In [14]:
def extract_first_name_for_gender(author_string):
    """Extract FIRST NAME ONLY from author string for gender inference"""
    if pd.isna(author_string) or author_string == "[]" or author_string == "['']":
        return None
    
    # 1. Get the clean full name using our previous function
    full_name = clean_author_names(author_string)
    if not full_name:
        return None
    
    # 2. Extract first name (first word after cleaning)
    first_name = full_name.split()[0] if full_name.split() else None
    
    # 3. Special handling for initials/abbreviations
    if first_name and len(first_name) <= 2:
        # If first "name" is an initial like "J." or "JK", try to get next word
        name_parts = full_name.split()
        if len(name_parts) > 1:
            first_name = name_parts[1]  # Try second word
    
    # 4. Final cleanup: remove any remaining punctuation
    if first_name:
        first_name = re.sub(r'[^\w\s]', '', first_name)
    
    return first_name if first_name and len(first_name) > 1 else None

In [15]:
def clean_authors_categories(books_df):
    
    books_authors = books_df.copy()

    # Clean authors - keep first author only for gender analysis
    books_authors['author_first_name'] = books_authors['authors'].apply(clean_author_names)
    books_authors['author_first_name'] = books_authors['author_first_name'].apply(extract_first_name_for_gender)

    # Clean categories - take first category
    books_authors['primary_category'] = (
    books_authors['categories']
    .str.strip("[]'\" ")  # Remove brackets and quotes
    .str.split('|')       # Split multiple categories
    .str[0]               # Take first category
    .str.strip("'\" ")    # Clean any remaining quotes
)
    
    return books_authors

In [16]:
books_clean=clean_authors_categories(books_critical)

In [17]:
# Extract year from all date formats
def extract_year(date_str):
    """
    Extract year from various date formats:
    - '2005-01-01' -> '2005'
    - '2005-02' -> '2005'
    - '1996' -> '1996'
    - '2000-04-26' -> '2000'
    """
    if pd.isna(date_str):
        return None
    
    # Convert to string if it's not already
    date_str = str(date_str).strip()
    
    # If it's already just a year (4 digits)
    if date_str.isdigit() and len(date_str) == 4:
        return date_str
    
    # Try to extract year from various formats
    # Look for 4 digits at the beginning of the string
    import re
    year_match = re.search(r'^(\d{4})', date_str)
    if year_match:
        return year_match.group(1)
    
    return None


In [18]:
# Apply the extraction
books_clean['Year'] = books_clean['publishedDate'].apply(extract_year)
# Convert to integer (this will handle NaN properly)
books_clean['Year'] = books_clean['Year'].astype('Int64')

In [19]:
# Create decade column
books_clean['Decade'] = (books_clean['Year'] // 10) * 10

In [20]:
books_clean.to_csv('books_clean.csv', index=False)
reviews_clean.to_csv('reviews_clean.csv', index=False)